In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("silero_all_thresholds_predictions.csv")

dev = df[
    df["split"].astype("string").str.strip().str.lower().eq("dev")
].copy()

dev["speech_true"] = (
    dev["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("yes")
)

thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

rows = []

for threshold in thresholds:
    suffix = f"{threshold:.2f}"
    col = f"vad_speech_detected_{suffix}"

    valid = dev[
        dev[col].notna()
        & dev["speech_present"].notna()
    ].copy()

    y_true = valid["speech_true"].astype(bool)

    y_pred = (
        valid[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

    keep = y_pred.notna()
    y_true = y_true[keep]
    y_pred = y_pred[keep].astype(bool)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[False, True]
    ).ravel()

    rows.append({
        "threshold": threshold,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "fp": fp,
        "fn": fn,
    })

vad_metrics = pd.DataFrame(rows)

for col in ["accuracy", "precision", "recall", "f1"]:
    vad_metrics[col] = vad_metrics[col] * 100

print(
    vad_metrics.to_string(
        index=False,
        formatters={
            "threshold": "{:.2f}".format,
            "accuracy": "{:.2f}".format,
            "precision": "{:.2f}".format,
            "recall": "{:.2f}".format,
            "f1": "{:.2f}".format,
        }
    )
)





In [ ]:

import pandas as pd

vad = pd.read_csv("silero_all_thresholds_predictions.csv")
whisper = pd.read_csv("whisper_medium_predictions.csv")

vad = vad[
    vad["split"].astype("string").str.strip().str.lower().eq("dev")
].copy()

whisper_pred = (
    whisper[["file_id", "whisper_med_prediction"]]
    .drop_duplicates("file_id")
)

df = vad.merge(
    whisper_pred,
    on="file_id",
    how="left"
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["true_language"] = (
    df["langs_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["pred_language"] = (
    df["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["target_language"] = df["true_language"].replace({
    "mandarin": "chinese"
})

no_speech = df["speech_true"].eq("no")

valid_speech = (
    df["speech_true"].eq("yes")
    & df["true_language"].notna()
    & df["true_language"].ne("")
    & ~df["true_language"].str.contains(";", na=False)
    & ~df["true_language"].isin(
        {"neutral", "unclear", "mixed", "cantonese"}
    )
    & ~df["speech_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("electronic")
)

analysis = df[
    no_speech | valid_speech
].copy()


raw_whisper_correct = (
    analysis["speech_true"].eq("yes")
    & (
        analysis["pred_language"]
        == analysis["target_language"]
    )
)

raw_whisper_accuracy = raw_whisper_correct.mean()

print(
    f"Raw Whisper-medium end-to-end accuracy: "
    f"{raw_whisper_accuracy * 100:.2f}%"
)

thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

rows = []

for threshold in thresholds:
    suffix = f"{threshold:.2f}"
    vad_col = f"vad_speech_detected_{suffix}"

    vad_pass = (
        analysis[vad_col]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

    valid = vad_pass.notna()

    temp = analysis[valid].copy()
    vad_pass = vad_pass[valid].astype(bool)

    is_no_speech = temp["speech_true"].eq("no")
    is_speech = temp["speech_true"].eq("yes")

    lid_correct = (
        temp["pred_language"] == temp["target_language"]
    )

    end_to_end_correct = (
        (is_no_speech & ~vad_pass)
        |
        (is_speech & vad_pass & lid_correct)
    )

    rows.append({
        "threshold": threshold,
        "n": len(temp),
        "correct": int(end_to_end_correct.sum()),
        "end_to_end_accuracy": end_to_end_correct.mean() * 100,
        "speech_passed": int((is_speech & vad_pass).sum()),
        "speech_rejected": int((is_speech & ~vad_pass).sum()),
        "no_speech_rejected": int((is_no_speech & ~vad_pass).sum()),
        "no_speech_passed": int((is_no_speech & vad_pass).sum()),
    })

e2e_results = pd.DataFrame(rows)

print(
    e2e_results.to_string(
        index=False,
        formatters={
            "threshold": "{:.2f}".format,
            "end_to_end_accuracy": "{:.2f}".format,
        }
    )
)


In [ ]:
import pandas as pd
import numpy as np

VAD_THRESHOLD = 0.20
CONF_THRESHOLDS = [
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90, 0.95
]

vad = pd.read_csv("silero_all_thresholds_predictions.csv")
whisper = pd.read_csv("whisper_medium_predictions.csv")

whisper_cols = (
    whisper[
        [
            "file_id",
            "whisper_med_prediction",
            "whisper_med_confidence"
        ]
    ]
    .drop_duplicates("file_id")
)

df = vad.merge(
    whisper_cols,
    on="file_id",
    how="left"
)

df["split_clean"] = (
    df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["true_language"] = (
    df["langs_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["pred_language"] = (
    df["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["target_language"] = (
    df["true_language"]
    .replace({"mandarin": "chinese"})
)

vad_col = f"vad_speech_detected_{VAD_THRESHOLD:.2f}"

df["vad_pass"] = (
    df[vad_col]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
)

no_speech = df["speech_true"].eq("no")

valid_speech = (
    df["speech_true"].eq("yes")
    & df["true_language"].notna()
    & df["true_language"].ne("")
    & ~df["true_language"].str.contains(";", na=False)
    & ~df["true_language"].isin(
        {"neutral", "unclear", "mixed"}
    )
    & ~df["speech_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("electronic")
)

analysis = df[
    (no_speech | valid_speech)
    & df["vad_pass"].notna()
].copy()

analysis["vad_pass"] = analysis["vad_pass"].astype(bool)

analysis["lid_correct"] = (
    analysis["pred_language"]
    == analysis["target_language"]
)


def evaluate_confidence_gate(data, threshold):

    auto_no_speech = ~data["vad_pass"]

    chinese_prediction = (
        data["pred_language"].eq("chinese")
    )

    auto_language = (
        data["vad_pass"]
        & ~chinese_prediction
        & data["whisper_med_confidence"].ge(threshold)
    )

    manual_review = (
        data["vad_pass"]
        & (
            chinese_prediction
            | data["whisper_med_confidence"].lt(threshold)
        )
    )

    auto_no_speech_correct = (
        auto_no_speech
        & data["speech_true"].eq("no")
    )

    auto_language_correct = (
        auto_language
        & data["speech_true"].eq("yes")
        & data["lid_correct"]
    )

    auto_resolved = auto_no_speech | auto_language
    auto_correct = (
        auto_no_speech_correct
        | auto_language_correct
    )

    language_n = int(auto_language.sum())
    language_correct_n = int(auto_language_correct.sum())

    return {
        "confidence_threshold": threshold,
        "n": len(data),

        "accepted_language_n": language_n,

        "accepted_language_accuracy":
            language_correct_n / language_n * 100
            if language_n > 0 else np.nan,

        "auto_no_speech_n":
            int(auto_no_speech.sum()),

        "auto_resolved_n":
            int(auto_resolved.sum()),

        "auto_resolved_pct":
            auto_resolved.mean() * 100,

        "manual_review_n":
            int(manual_review.sum()),

        "manual_review_pct":
            manual_review.mean() * 100,

        "overall_auto_accuracy":
            auto_correct.sum() / auto_resolved.sum() * 100
            if auto_resolved.sum() > 0 else np.nan
    }


dev = analysis[
    analysis["split_clean"].eq("dev")
].copy()

dev_results = pd.DataFrame([
    evaluate_confidence_gate(dev, t)
    for t in CONF_THRESHOLDS
])

print("\nDEVELOPMENT CONFIDENCE GATING")
print(
    dev_results.to_string(
        index=False,
        formatters={
            "confidence_threshold": "{:.2f}".format,
            "accepted_language_accuracy": "{:.2f}".format,
            "auto_resolved_pct": "{:.2f}".format,
            "manual_review_pct": "{:.2f}".format,
            "overall_auto_accuracy": "{:.2f}".format,
        }
    )
)

eligible = dev_results[
    dev_results["accepted_language_accuracy"] >= 90
]

if len(eligible) == 0:
    raise ValueError(
        "No confidence threshold achieved 90% "
        "accepted-language accuracy on dev."
    )

selected_threshold = float(
    eligible.iloc[0]["confidence_threshold"]
)

print(
    f"\nSelected confidence threshold: "
    f"{selected_threshold:.2f}"
)


eval_df = analysis[
    analysis["split_clean"].eq("eval")
].copy()

eval_result = evaluate_confidence_gate(
    eval_df,
    selected_threshold
)

print("\nHELD-OUT EVALUATION")
for key, value in eval_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")


is_no_speech = eval_df["speech_true"].eq("no")
is_speech = eval_df["speech_true"].eq("yes")

raw_whisper_correct = (
    is_speech
    & eval_df["lid_correct"]
)

vad_pipeline_correct = (
    (is_no_speech & ~eval_df["vad_pass"])
    |
    (
        is_speech
        & eval_df["vad_pass"]
        & eval_df["lid_correct"]
    )
)

print(
    f"\nRaw Whisper end-to-end accuracy: "
    f"{raw_whisper_correct.mean() * 100:.2f}%"
)

print(
    f"VAD + Whisper end-to-end accuracy: "
    f"{vad_pipeline_correct.mean() * 100:.2f}%"
)


In [ ]:
import pandas as pd
import numpy as np

VAD_THRESHOLD = 0.20
CONF_THRESHOLD = 0.70

vad = pd.read_csv("silero_all_thresholds_predictions.csv")
whisper = pd.read_csv("whisper_medium_predictions.csv")

whisper_cols = (
    whisper[
        [
            "file_id",
            "whisper_med_prediction",
            "whisper_med_confidence"
        ]
    ]
    .drop_duplicates("file_id")
)

df = vad.merge(
    whisper_cols,
    on="file_id",
    how="left"
)

df["split_clean"] = (
    df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["pred_language"] = (
    df["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

def parse_languages(x):
    if pd.isna(x):
        return set()

    langs = {
        str(lang).strip().lower()
        for lang in str(x).split(";")
        if str(lang).strip()
    }

    langs -= {"neutral", "mixed", "unclear"}

    return {
        "chinese" if lang == "mandarin" else lang
        for lang in langs
    }

df["true_language_set"] = (
    df["langs_present"]
    .apply(parse_languages)
)

df["n_valid_languages"] = (
    df["true_language_set"]
    .map(len)
)

vad_col = f"vad_speech_detected_{VAD_THRESHOLD:.2f}"

df["vad_pass"] = (
    df[vad_col]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

eligible = (
    df["speech_true"].eq("no")
    |
    (
        df["speech_true"].eq("yes")
        & df["n_valid_languages"].ge(1)
        & ~df["speech_type"]
            .astype("string")
            .str.strip()
            .str.lower()
            .eq("electronic")
    )
)

analysis = df[
    eligible
    & df["vad_pass"].notna()
].copy()

analysis["vad_pass"] = (
    analysis["vad_pass"]
    .astype(bool)
)

analysis["single_language"] = (
    analysis["speech_true"].eq("yes")
    & analysis["n_valid_languages"].eq(1)
)

analysis["multilingual"] = (
    analysis["speech_true"].eq("yes")
    & analysis["n_valid_languages"].ge(2)
)

analysis["language_matches_any_reference"] = (
    analysis.apply(
        lambda r:
            r["pred_language"] in r["true_language_set"],
        axis=1
    )
)

analysis["chinese_prediction"] = (
    analysis["pred_language"].eq("chinese")
)

analysis["auto_no_speech"] = (
    ~analysis["vad_pass"]
)

analysis["auto_language"] = (
    analysis["vad_pass"]
    & ~analysis["chinese_prediction"]
    & analysis["whisper_med_confidence"].ge(CONF_THRESHOLD)
)

analysis["manual_review"] = (
    analysis["vad_pass"]
    & (
        analysis["chinese_prediction"]
        | analysis["whisper_med_confidence"].lt(CONF_THRESHOLD)
    )
)

for split in ["dev", "eval"]:

    x = analysis[
        analysis["split_clean"].eq(split)
    ].copy()

    no_speech = x[
        x["speech_true"].eq("no")
    ]

    single = x[
        x["single_language"]
    ]

    multi = x[
        x["multilingual"]
    ]

    auto_no_speech_correct = (
        x["auto_no_speech"]
        & x["speech_true"].eq("no")
    )

    auto_single = (
        x["auto_language"]
        & x["single_language"]
    )

    auto_single_correct = (
        auto_single
        & x["language_matches_any_reference"]
    )

    auto_multi = (
        x["auto_language"]
        & x["multilingual"]
    )

    deployment_auto_resolved = (
        x["auto_no_speech"]
        | x["auto_language"]
    )

    deployment_correct = (
        auto_no_speech_correct
        | auto_single_correct
    )

    print("\n" + "=" * 60)
    print(split.upper())
    print("=" * 60)

    print(f"Eligible clips: {len(x)}")
    print(f"No-speech clips: {len(no_speech)}")
    print(f"Single-language clips: {len(single)}")
    print(f"Multilingual clips: {len(multi)}")

    print("\nSINGLE-LANGUAGE")
    print(
        f"Auto-accepted: "
        f"{auto_single.sum()}"
    )

    if auto_single.sum() > 0:
        print(
            f"Accepted-language accuracy: "
            f"{auto_single_correct.sum() / auto_single.sum() * 100:.2f}%"
        )

    print("\nMULTILINGUAL")
    print(
        f"Sent to manual review: "
        f"{multi['manual_review'].sum()} / {len(multi)} "
        f"({multi['manual_review'].mean() * 100:.2f}%)"
    )

    print(
        f"Auto-accepted as one language: "
        f"{auto_multi.sum()} / {len(multi)} "
        f"({auto_multi.sum() / len(multi) * 100:.2f}%)"
    )

    print(
        f"Rejected by VAD as no speech: "
        f"{multi['auto_no_speech'].sum()} / {len(multi)} "
        f"({multi['auto_no_speech'].mean() * 100:.2f}%)"
    )

    if auto_multi.sum() > 0:
        print(
            f"Among auto-accepted multilingual clips, "
            f"top-1 matched at least one reference: "
            f"{multi.loc[multi['auto_language'], 'language_matches_any_reference'].mean() * 100:.2f}%"
        )

    print("\nFULL DEPLOYMENT")

    print(
        f"Automatically handled: "
        f"{deployment_auto_resolved.sum()} / {len(x)} "
        f"({deployment_auto_resolved.mean() * 100:.2f}%)"
    )

    print(
        f"Manual review: "
        f"{x['manual_review'].sum()} / {len(x)} "
        f"({x['manual_review'].mean() * 100:.2f}%)"
    )

    if deployment_auto_resolved.sum() > 0:
        print(
            f"Accuracy among automatic decisions: "
            f"{deployment_correct.sum() / deployment_auto_resolved.sum() * 100:.2f}%"
        )


In [ ]:

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("silero_all_thresholds_predictions.csv")

df["split_clean"] = (
    df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

vad_col = "vad_speech_detected_0.20"

df["vad_pred"] = (
    df[vad_col]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

eval_df = df[
    df["split_clean"].eq("eval")
    & df["speech_true"].isin(["yes", "no"])
    & df["vad_pred"].notna()
].copy()

y_true = eval_df["speech_true"].eq("yes")
y_pred = eval_df["vad_pred"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

tp = ((y_true == True) & (y_pred == True)).sum()
tn = ((y_true == False) & (y_pred == False)).sum()
fp = ((y_true == False) & (y_pred == True)).sum()
fn = ((y_true == True) & (y_pred == False)).sum()

print(f"N = {len(eval_df)}")
print(f"Accuracy = {accuracy * 100:.2f}%")
print(f"Precision = {precision * 100:.2f}%")
print(f"Recall = {recall * 100:.2f}%")
print(f"F1 = {f1:.4f}")
print()
print(f"TP = {tp}")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
